In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * DATABASE DESIGN — From Flat Tables to Production-Ready Models
 * ─────────────────────────────────────────────────────────────────────
 * 1.  RAW DATA PAIN .................... Flat tables are an anti-pattern
 * 2.  SAMPLE TABLE ..................... raw_learning_events
 * 3.  THE REAL PROBLEMS ................ Inconsistency, duplication, confusion
 * 4.  STUDENT DETAILS REPEATED ......... Name inconsistency, update nightmare
 * 5.  COURSE DETAILS REPEATED .......... Title changes = 50,000 row updates
 * 6.  DUPLICATE ENROLLMENT ............. No business rule enforcement
 * 7.  SPREADSHEET vs RELATIONAL ........ Mindset shift
 * 8.  CORE CONCEPTS .................... Entities, Attributes, Relationships
 * 9.  NORMALIZATION .................... Reduce duplication, prevent anomalies
 * 10. ENTITIES ......................... Real-world things you store data about
 * 11. ATTRIBUTES ....................... Properties that describe entities
 * 12. RELATIONSHIPS .................... How entities interact via keys
 * 13. CARDINALITY ...................... 1:1, 1:N, N:M
 * 14. OPTIONALITY ...................... Mandatory vs Optional relationships
 * 15. MENTAL MODEL ..................... Design for Truth, not Convenience
 * 16. AFFILIATE EXAMPLE ................ Derived value trap
 * 17. TRUTH TABLE ...................... Single source of truth
 * 18. CONVENIENCE TABLE TRAP ........... Derive from truth, don't replace it
 * 19. SCHEMA DESIGN .................... Surrogate keys, constraints, bridges
 * 20. KEYS ............................. PK, FK, Composite, Surrogate, Natural
 * 21. NORMALIZATION DEEP DIVE .......... 1NF, 2NF, 3NF
 * 22. ANOMALIES ........................ Update, Insert, Delete
 * ─────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * Part-1: RAW DATA PAIN — Why Flat Tables Are a Database Anti-Pattern
 */

/**
 * 1. THE COMFORT OF FLAT TABLES
 * ─────────────────────────────

 *   Shuruaat mein flat table bahut convenient lagti hai.
 *   Ek spreadsheet export karo, ek table mein dump karo.
 *   Rows dikhti hain, values dikhti hain — sab thik lagta hai.
 *   Par ACTUALLY tum store kar rahe ho:
 *     → Structure nahi — REPETITION
 *     → Relationships nahi — COPIES
 *     → Clarity nahi — future ka CONFUSION
 *
 * THE FLAT TABLE ILLUSION:
 *   raw_learning_events ek baar mein sab kuch deta dikhi:
 *     → Student info  ✓
 *     → Course info   ✓
 *     → Instructor    ✓
 *     → Payment       ✓
 *     → Timestamps    ✓
 *
 *   Sab ek jagah — YAHI toh PROBLEM hai.
 *
 *   SQL is NOT about storing everything.
 *   SQL IS about structuring RELATIONSHIPS correctly.
 */


/**
 * 2. SAMPLE TABLE — raw_learning_events (The Anti-Pattern)
 * ─────────────────────────────────────────────────────────
 * Hinglish:
 *   Ye table ek learning platform ka example hai.
 *   Har row = "ek learning event" (kisi ne koi course khareeda).
 *   Problem: ek hi row mein Student + Course + Instructor + Payment sab hai.
 *
 * ┌────────┬─────────────────────┬──────────────┬────────────┬──────────────────────┬──────────────────┬──────────────────┬──────────────┬──────────┬───────────────┬────────────────┬─────────────────────┬─────────────────────┐
 * │ row_id │ student_email       │ student_name │ course_code│ course_title         │ instructor_email │ instructor_name  │ listed_price │ currency │ payment_id    │ payment_status │ paid_at             │ enrolled_at         │
 * ├────────┼─────────────────────┼──────────────┼────────────┼──────────────────────┼──────────────────┼──────────────────┼──────────────┼──────────┼───────────────┼────────────────┼─────────────────────┼─────────────────────┤
 * │ 1      │ asha@gmail.com      │ Asha         │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1001      │ PAID           │ 2026-01-05 10:10:00 │ 2026-01-05 10:12:00 │
 * │ 2      │ asha@gmail.com      │ Asha Sharma  │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1002      │ REFUNDED       │ 2026-01-06 09:00:00 │ 2026-01-06 09:02:00 │
 * │ 3      │ ravi@gmail.com      │ Ravi         │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 599.00       │ INR      │ pay_1003      │ PAID           │ 2026-01-10 18:00:00 │ 2026-01-10 18:02:00 │
 * │ 4      │ neha@gmail.com      │ Neha         │ DE201      │ Data Eng Found       │ sara@tuf.com     │ Sara             │ 19.00        │ USD      │ pay_1004      │ PAID           │ 2026-01-12 20:00:00 │ 2026-01-12 20:05:00 │
 * │ 5      │ neha@gmail.com      │ Neha         │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1005      │ PENDING        │ NULL                │ 2026-01-15 08:00:00 │
 * │ 6      │ dev@gmail.com       │ Dev          │ LLD301     │ LLD Bootcamp         │ mike@tuf.com     │ Mike             │ 49.00        │ USD      │ pay_1006      │ PAID           │ 2026-01-20 11:30:00 │ 2026-01-20 11:31:00 │
 * │ 7      │ dev@gmail.com       │ Dev          │ LLD301     │ LLD Bootcamp         │ mike@tuf.com     │ Mike             │ 49.00        │ USD      │ pay_1007      │ PAID           │ 2026-01-21 12:00:00 │ 2026-01-21 12:01:00 │
 * │ 8      │ utkarsh@gmail.com   │ Utkarsh      │ DSA401     │ DSA Mastery          │ raj@tuf.com      │ Raj              │ 999.00       │ INR      │ pay_1008      │ PAID           │ 2026-01-22 09:00:00 │ 2026-01-22 09:05:00 │
 * │ 9      │ anuj@gmail.com      │ Anuj         │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1009      │ PAID           │ 2026-01-25 10:00:00 │ 2026-01-25 10:02:00 │
 * │ 10     │ ronit@gmail.com     │ Ronit        │ DE201      │ Data Eng Found       │ sara@tuf.com     │ Sara             │ 19.00        │ USD      │ pay_1010      │ PAID           │ 2026-01-25 15:00:00 │ 2026-01-25 15:05:00 │
 * │ 11     │ shivi@gmail.com     │ Shivi        │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1011      │ PAID           │ 2026-01-28 19:00:00 │ 2026-01-28 19:02:00 │
 * │ 12     │ siddhant@gmail.com  │ Siddhant     │ DE201      │ Data Eng Found       │ sara@tuf.com     │ Sara             │ 19.00        │ USD      │ pay_1012      │ REFUNDED       │ 2026-01-29 10:00:00 │ 2026-01-29 10:01:00 │
 * │ 13     │ vinayak@gmail.com   │ Vinayak      │ DSA401     │ DSA Mastery          │ raj@tuf.com      │ Raj              │ 999.00       │ INR      │ pay_1013      │ PAID           │ 2026-02-01 09:00:00 │ 2026-02-01 09:03:00 │
 * │ 14     │ mridul@gmail.com    │ Mridul       │ LLD301     │ LLD Bootcamp         │ mike@tuf.com     │ Mike             │ 49.00        │ USD      │ pay_1014      │ PAID           │ 2026-02-02 14:00:00 │ 2026-02-02 14:02:00 │
 * │ 15     │ prashant@gmail.com  │ Prashant     │ SQL101     │ SQL Basics           │ raj@tuf.com      │ Raj              │ 499.00       │ INR      │ pay_1015      │ PAID           │ 2026-02-03 16:00:00 │ 2026-02-03 16:01:00 │
 * └────────┴─────────────────────┴──────────────┴────────────┴──────────────────────┴──────────────────┴──────────────────┴──────────────┴──────────┴───────────────┴────────────────┴─────────────────────┴─────────────────────┘
 *
 * Entities mixed in this one table:
 * ┌──────────────────┬────────────────────────────────────────────────┐
 * │ Entity Type      │ Columns stored                                 │
 * ├──────────────────┼────────────────────────────────────────────────┤
 * │ Student          │ student_email, student_name                    │
 * │ Course           │ course_code, course_title                      │
 * │ Instructor       │ instructor_email, instructor_name              │
 * │ Price            │ listed_price, currency                         │
 * │ Payment          │ payment_id, payment_status, paid_at            │
 * │ Enrollment       │ enrolled_at                                    │
 * └──────────────────┴────────────────────────────────────────────────┘
 */


/**
 * 3. THE REAL PROBLEMS — Inconsistency, Duplication, Confusion
 * ─────────────────────────────────────────────────────────────
 * Hinglish:
 *   Is table mein 3 bade problems hain:
 *   1. Asha ka naam 2 rows mein alag-alag hai — kaun sahi hai?
 *   2. SQL101 ki details 7 rows mein repeat ho rahi hain — agar title change ho?
 *   3. Dev ne LLD301 mein 2 baar payment ki — duplicate ya bug?
 *
 * ┌──────────────────────────────────────────────────────────────────────────┐
 * │ PROBLEM 1: Student details repeated                                      │
 * │   Row 1: 'asha@gmail.com', 'Asha'                                        │
 * │   Row 2: 'asha@gmail.com', 'Asha Sharma'   ← same email, different name  │
 * │   Kaunsa sahi hai? DB kuch nahi jaanta.                                  │
 * ├──────────────────────────────────────────────────────────────────────────┤
 * │ PROBLEM 2: Course details repeated                                       │
 * │   SQL101 ki rows: 1, 2, 3, 5, 9, 11, 15 = 7 rows                         │
 * │   Title "SQL Basics" → "SQL Fundamentals" → 7 rows update karo           │
 * │   50,000 purchases hote toh → 50,000 rows update!                        │
 * ├──────────────────────────────────────────────────────────────────────────┤
 * │ PROBLEM 3: Duplicate enrollment                                          │
 * │   Row 6: dev@gmail.com, LLD301, pay_1006, PAID                           │
 * │   Row 7: dev@gmail.com, LLD301, pay_1007, PAID                           │
 * │   Same student, same course, 2 successful payments — kya hua?            │
 * │   Flat table enforce nahi kar sakta: "ek hi enrollment per student"      │
 * └──────────────────────────────────────────────────────────────────────────┘
 */


/**
 * 4. STUDENT DETAILS REPEATED — Name Inconsistency, Update Nightmare
 * ────────────────────────────────────────────────────────────────────
 * Hinglish:
 *   Asha ka naam 2 jagah hai — 'Asha' aur 'Asha Sharma'.
 *   Query karo toh dono rows aayengi. Kaunsi correct?
 *   Update karo toh SABHI rows update karne padenge — koi bhi miss hua toh inconsistency.
 *   Ye hai "No Single Source of Truth" ka problem.
 *
 * Query that exposes the problem:
 *
 *   SELECT student_email, student_name, COUNT(*)
 *   FROM raw_learning_events
 *   WHERE student_email = 'asha@gmail.com'
 *   GROUP BY student_email, student_name;
 *
 *   Output:
 *   +-----------------+--------------+-------+
 *   | student_email   | student_name | COUNT |
 *   +-----------------+--------------+-------+
 *   | asha@gmail.com  | Asha         | 1     |
 *   | asha@gmail.com  | Asha Sharma  | 1     |   ← same user, 2 names!
 *   +-----------------+--------------+-------+
 *
 * Consequences:
 *   1. Customer support confusion — "Aapka system mujhe kab 'Asha' aur kab 'Asha Sharma' bulata hai?"
 *   2. Reporting errors — COUNT by name → Asha 2 baar giни jaayegi
 *   3. Update nightmare → sab rows update karo WHERE email = 'asha@gmail.com'
 *   4. No single source of truth — student info scattered across all enrollment rows
 *
 * Relational Solution:
 *   students(id, email, name) → ek student, ek row
 *   Enrollments reference student_id — naam sirf ek jagah
 *   → naam change: 1 row update, done.
 */


/**
 * 5. COURSE DETAILS REPEATED — Title Changes = 50,000 Row Updates
 * ─────────────────────────────────────────────────────────────────
 * Hinglish:
 *   SQL101 ki details — title, instructor, price — har enrollment row mein copy hoti hai.
 *   50,000 log enrolled hain → title change karo → 50,000 rows update karo!
 *   Table lock hogi, time lagega, partial update ka risk.
 *
 * With flat table (WRONG):
 *
 *   UPDATE raw_learning_events
 *   SET course_title = 'SQL Fundamentals',
 *       listed_price = 599.00
 *   WHERE course_code = 'SQL101';
 *   → 50,000 rows update, takes minutes, locks table, risk of partial updates!
 *
 * Relational Solution:
 *
 *   courses(id, code, title, price, instructor_id) → ek course, ek row
 *
 *   UPDATE courses SET title = 'SQL Fundamentals' WHERE code = 'SQL101';
 *   → 1 row update, instant, all enrollments see new title via JOIN
 *
 *   This is the POWER OF NORMALIZATION.
 */


/**
 * 6. DUPLICATE ENROLLMENT — No Business Rule Enforcement
 * ───────────────────────────────────────────────────────
 * Hinglish:
 *   Dev ne LLD301 mein 2 baar payment ki — dono PAID hain.
 *   Ye duplicate enrollment hai? Payment retry? System bug?
 *   Flat table kuch enforce nahi kar sakta.
 *   Na UNIQUE constraint, na business rule — DB blindly accept karta hai.
 *
 * Query that exposes the problem:
 *
 *   SELECT * FROM raw_learning_events
 *   WHERE student_email = 'dev@gmail.com'
 *     AND course_code   = 'LLD301';
 *
 *   Output:
 *   Row 6: dev@gmail.com, LLD301, pay_1006, PAID  ← dono PAID
 *   Row 7: dev@gmail.com, LLD301, pay_1007, PAID  ← duplicate!
 *
 * Flat table CANNOT enforce:
 *   → UNIQUE on (student_email, course_code)
 *   → "One active enrollment per student per course"
 *   → Payment validation
 *
 * Consequences:
 *   → Wrong reports ("kitne students ne LLD301 liya?" → Dev 2 baar count)
 *   → Customer confusion (Dev ko double charge ho gaya)
 *   → Support tickets
 *
 * Relational Solution:
 *   enrollments table:
 *     UNIQUE(student_id, course_id)
 *   → Database khud duplicate enrollment ko reject karta hai
 *   → Koi application logic nahi chahiye
 */


/**
 * 7. SPREADSHEET vs RELATIONAL THINKING — Mindset Shift
 * ──────────────────────────────────────────────────────
 * Hinglish:
 *   Ye table database design nahi hai — ye ek spreadsheet dump hai.
 *   Spreadsheet mein sochte hain: "sab ek jagah rakh do."
 *   Relational DB mein sochte hain: "entities kaun hain, relationships kya hain?"
 *
 * ┌─────────────────────────────────────────────────────────────────────┐
 * │ SPREADSHEET THINKING          │ RELATIONAL THINKING                 │
 * ├───────────────────────────────┼─────────────────────────────────────┤
 * │ Events row by row store karo  │ Entities model karo                 │
 * │ Information freely repeat karo│ Ek entity = ek row, ek jagah        │
 * │ Ek badi table sab ke liye     │ Multiple tables, foreign keys       │
 * │ Initially fast aur easy       │ Consistency aur integrity focus     │
 * │ Relationships ka koi dhyan nahi│ Relationships explicitly enforced  │
 * │ Sab convenient lage pehle     │ Updates ek row mein hote hain       │
 * └───────────────────────────────┴─────────────────────────────────────┘
 *
 * The shift in mindset:
 *   From: "How do I store this event?"
 *   To:   "What entities exist? How are they related?"
 *
 *   From: "I can see everything at once."
 *   To:   "I can query relationships efficiently."
 *
 * This is WHY normalization exists.
 * This is WHY relational databases were invented.
 */

In [ ]:
/**
 * Part-2: CORE CONCEPTS — Entities, Attributes, Relationships
 */

/**
 * 8. WHY CORE CONCEPTS MATTER — Preventing Chaos Before It Starts
 * ───────────────────────────────────────────────────────────────
 * Hinglish:
 *   Database chaos sirf bura SQL likhne se nahi aata.
 *   Aata hai kyunki developers PAUSE nahi karte sochne ke liye:
 *   "Data ka structure kya hona chahiye?"
 *
 * When you IGNORE these concepts:
 *   → Entities mix ho jaate hain
 *   → Attributes duplicate ho jaate hain
 *   → Relationships unclear rehte hain
 *   → Constraints missing rehte hain
 *   → Result = DATA CHAOS
 *
 * When you APPLY these principles:
 *   → Har entity ki apni table
 *   → Har attribute ek jagah
 *   → Relationships enforced hain
 *   → Database naturally scale karta hai
 */


/**
 * 9. NORMALIZATION — Reduce Duplication, Prevent Anomalies
 * ─────────────────────────────────────────────────────────
 * Hinglish:
 *   Normalization = ek process jisme tables is tarah design karte hain ki
 *   duplication kam ho aur data anomalies prevent ho.
 *   Simple rule: har fact sirf ek baar store karo.
 *
 * What normalization PREVENTS:
 *   → Update anomaly: kuch rows update ho, baaki nahi
 *   → Insertion anomaly: insert karne ke liye redundant data chahiye
 *   → Deletion anomaly: ek row delete karo, aur data bhi chala jaata hai
 *
 * Flat table (raw_learning_events) stores:
 *   → Student details repeated on every enrollment
 *   → Course details repeated on every enrollment
 *   → Instructor details repeated on every enrollment
 *
 * Normalized design:
 *   students(id, email, name)              → 1 row per student
 *   courses(id, code, title, price)        → 1 row per course
 *   instructors(id, email, name)           → 1 row per instructor
 *   enrollments(id, student_id, course_id, enrolled_at)
 *   payments(id, enrollment_id, amount, status, paid_at)
 *
 * Results:
 *   → Title update  → 1 row
 *   → Name update   → 1 row
 *   → History safe when student deleted
 *
 * NORMALIZATION IS NOT "SPLITTING TABLES RANDOMLY."
 * IT IS DESIGNING SO THAT EACH FACT IS STORED ONCE.
 */


/**
 * 10. ENTITIES — Real-World Things You Store Data About
 * ──────────────────────────────────────────────────────
 * Hinglish:
 *   Entity = ek real-world cheez jiske baare mein tum data store karte ho.
 *   Test: "Main is cheez ke baare mein information store karna chahta hoon."
 *   Agar answer ek noun hai — yahi entity hai.
 *
 * In our learning platform:
 * ┌─────────────────┬────────────────────────────────────────────────┐
 * │ Entity          │ What it represents                             │
 * ├─────────────────┼────────────────────────────────────────────────┤
 * │ Student         │ A person who learns                            │
 * │ Course          │ A learning offering                            │
 * │ Instructor      │ A person who teaches                           │
 * │ Payment         │ A financial transaction                        │
 * │ Enrollment      │ The act of a student joining a course          │
 * └─────────────────┴────────────────────────────────────────────────┘
 *
 * KEY INSIGHT — Entities exist INDEPENDENTLY:
 *   → Student exists whether they pay or not
 *   → Course exists whether someone enrolls or not
 *   → Payment exists BECAUSE an enrollment exists (dependency)
 *
 * GOOD DATABASE DESIGN = Identify entities first.
 * Each entity gets its OWN TABLE.
 */


/**
 * 11. ATTRIBUTES — Properties That Describe Entities
 * ────────────────────────────────────────────────────
 * Hinglish:
 *   Attribute = entity ki koi property ya characteristic.
 *   Attributes entity ko describe karte hain — khud se exist nahi karte.
 *   "email" koi cheez nahi hai — ye student ki property hai.
 *   "price" koi cheez nahi hai — ye course ki property hai.
 *
 * ┌─────────────────┬─────────────────────────────────────────────────┐
 * │ Entity          │ Attributes (Columns)                            │
 * ├─────────────────┼─────────────────────────────────────────────────┤
 * │ Student         │ id, email, name, phone, date_of_birth, created_at│
 * │ Course          │ id, code, title, price, currency, instructor_id  │
 * │ Instructor      │ id, email, name, joined_at                      │
 * │ Enrollment      │ id, student_id, course_id, enrolled_at, status  │
 * │ Payment         │ id, enrollment_id, amount, status, paid_at      │
 * └─────────────────┴─────────────────────────────────────────────────┘
 *
 * Correct attribute placement:
 *   ✅ email → attribute of Student
 *   ✅ title → attribute of Course
 *   ✅ amount → attribute of Payment
 *
 *   ❌ email should NOT be stored in Enrollment
 *   ❌ course_title should NOT be stored in Payment
 *   ❌ instructor_name should NOT be stored in Course
 *
 * In database design:
 *   Attributes → COLUMNS in the entity's table.
 */


/**
 * 12. RELATIONSHIPS — How Entities Interact via Keys
 * ────────────────────────────────────────────────────
 * Hinglish:
 *   Relationship = ek entity ka doosri entity se connection.
 *   Ye tab hota hai jab ek table ka PRIMARY KEY doosri table mein
 *   FOREIGN KEY ki tarah store hota hai.
 *
 * Example 1: Course → Instructor (1:N)
 *
 *   instructors table:
 *   ┌───────────────┬─────────────────┐
 *   │ instructor_id │ full_name       │
 *   │ (PRIMARY KEY) │                 │
 *   ├───────────────┼─────────────────┤
 *   │ 1             │ Raj             │
 *   │ 2             │ Sara            │
 *   └───────────────┴─────────────────┘
 *                       ↑
 *   courses table:       │ (FOREIGN KEY)
 *   ┌───────────┬─────────────┬────────────────┐
 *   │ course_id │ title       │ instructor_id  │
 *   ├───────────┼─────────────┼────────────────┤
 *   │ 1         │ SQL Basics  │ 1              │ ← Raj se connected
 *   │ 2         │ DSA Mastery │ 1              │ ← Raj se connected
 *   │ 3         │ LLD Design  │ 2              │ ← Sara se connected
 *   └───────────┴─────────────┴────────────────┘
 *
 * WITHOUT explicit relationships (flat table):
 *   → Relationships IMPLIED but not ENFORCED
 *   → DB cannot validate if student exists
 *   → DB cannot prevent orphan payments
 *
 * WITH explicit relationships (foreign keys):
 *   enrollments:
 *     student_id REFERENCES students(id)    → student hona zaroori
 *     course_id  REFERENCES courses(id)     → course hona zaroori
 *
 *   payments:
 *     enrollment_id REFERENCES enrollments(id)
 *
 *   Database becomes SELF-VALIDATING.
 */


/**
 * 13. CARDINALITY — 1:1, 1:N, N:M
 * ─────────────────────────────────
 * Hinglish:
 *   Cardinality ek simple sawaal ka jawab hai: "KITNE?"
 *   "Ek side ke liye doosri side mein kitne rows ho sakte hain?"
 *
 * ┌───────────────────┬────────────────────────────────────────────────┐
 * │ Type              │ Example in Learning Platform                   │
 * ├───────────────────┼────────────────────────────────────────────────┤
 * │ ONE-TO-ONE (1:1)  │ Student → Profile (ek profile ya zero)         │
 * │ ONE-TO-MANY (1:N) │ Instructor → Courses (ek teacher many courses) │
 * │ MANY-TO-MANY (N:M)│ Students ↔ Courses (students many courses lein)│
 * └───────────────────┴────────────────────────────────────────────────┘
 *
 * 1. ONE-TO-ONE (1:1)
 *    students ────── student_profiles
 *    Enforcement: FK must also be UNIQUE
 *
 *    CREATE TABLE student_profiles (
 *      student_id INT UNIQUE,   ← UNIQUE enforce karta hai 1:1
 *      FOREIGN KEY (student_id) REFERENCES students(id)
 *    );
 *    Without UNIQUE → becomes 1:N accidentally!
 *
 * 2. ONE-TO-MANY (1:N)
 *    Instructor (1) ──────► Courses (N)
 *
 *    RULE OF THUMB: FK "repeat" wali side mein jaati hai.
 *    Many courses can have same instructor
 *    → instructor_id is stored in courses table.
 *
 * 3. MANY-TO-MANY (N:M)
 *    WRONG (beginners often do this):
 *      students table: courses_enrolled = "1,3,5"  ← list in column!
 *      → Breaks First Normal Form
 *      → Indexing useless, joins break, update chaos
 *
 *    CORRECT → Bridge (Junction) Table:
 *      enrollments table:
 *      ┌────────────────┬───────────┬───────────┐
 *      │ enrollment_id  │ student_id│ course_id │
 *      ├────────────────┼───────────┼───────────┤
 *      │ 1              │ 1         │ 1         │
 *      │ 2              │ 1         │ 3         │
 *      │ 3              │ 2         │ 1         │
 *      └────────────────┴───────────┴───────────┘
 *      Har row = ONE FACT (student joined course)
 *      N:M → two 1:N relationships ban jaate hain
 *
 * PRODUCTION TIP:
 *   N:M relationships ALWAYS model using a third table.
 *   One extra table never harms you. Wrong modeling will.
 */


/**
 * 14. OPTIONALITY — Mandatory vs Optional Relationships
 * ──────────────────────────────────────────────────────
 * Hinglish:
 *   Optionality ka matlab: kya relationship ZAROORI hai ya optional?
 *   Ye control hota hai NULL vs NOT NULL se FK column mein.
 *
 * MANDATORY (NOT NULL):
 *   Har course ka instructor hona ZAROORI hai.
 *
 *   CREATE TABLE courses (
 *     ...
 *     instructor_id INT NOT NULL,     ← MANDATORY
 *     FOREIGN KEY (instructor_id) REFERENCES instructors(id)
 *   );
 *   → NULL value insert karo → ERROR!
 *
 * OPTIONAL (NULL allowed):
 *   Kuch students ka profile ho sakta hai, kuch ka nahi.
 *
 *   CREATE TABLE student_profiles (
 *     ...
 *     student_id INT,                 ← NULL allowed = optional
 *     FOREIGN KEY (student_id) REFERENCES students(id)
 *   );
 *   → NULL = "is profile ka koi student linked nahi"
 *
 * PRODUCTION RULE:
 *   NOT NULL  → business mein required hai
 *   NULL      → optional relationship
 *   DB storage layer pe enforce karo — trust mat karo application pe
 */

In [ ]:
/**
 * Part-3: MENTAL MODEL — Design for Truth, Not Convenience
 */

/**
 * 15. THE WRONG APPROACH — Designing for Queries, Not Truth
 * ──────────────────────────────────────────────────────────
 * Hinglish:
 *   Bahut log pehle QUERIES sochte hain:
 *   "Revenue kitna hua?" "Students kitne hain?"
 *   Phir us query ko easy banane ke liye tables design karte hain.
 *   YE GALAT APPROACH HAI.
 *
 * Why it's wrong:
 *   → Tum CONVENIENCE ke liye optimize kar rahe ho, CORRECTNESS ke liye nahi
 *   → Tum DERIVED values ko FACTS ki tarah store karte ho
 *   → Data corrupt ho sakta hai
 *
 * THE CORRECT MENTAL MODEL:
 *   Tables design karo TRUTH ke liye.
 *   Queries baad mein optimize ho sakti hain.
 *   Data integrity baad mein fix nahi hoti asaani se.
 */


/**
 * 16. THE AFFILIATE EXAMPLE — Convenience vs Correctness
 * ────────────────────────────────────────────────────────
 * Hinglish:
 *   Scenario: Har user ka ek affiliate code hai.
 *   Jab koi us code se purchase kare → affiliate ko 15% milta hai.
 *   "Total earnings track karo."
 *
 * THE WRONG (CONVENIENT) DESIGN:
 *
 *   CREATE TABLE users (
 *     user_id  SERIAL PRIMARY KEY,
 *     affiliate_code VARCHAR(50) UNIQUE,
 *     total_affiliate_earnings DECIMAL(10,2) DEFAULT 0   ← PROBLEM!
 *   );
 *
 *   -- Har purchase ke baad:
 *   UPDATE users
 *   SET total_affiliate_earnings = total_affiliate_earnings + 15%
 *   WHERE affiliate_code = 'RAJ123';
 *
 *   Easy. Clean. Fast.  BUT WRONG.
 *
 * WHY IT'S WRONG — agar ye ho toh?
 *   → Refund aaye           → column corrupt
 *   → Calculation bug       → galat total, verify nahi kar sakte
 *   → Commission 15% → 20%  → sab purani values galat
 *   → Audit chahiye         → impossible, no record of individual sales
 *
 *   Tum stored kiya DERIVED VALUE ko TRUTH ki tarah.
 *
 * THE CORRECT DESIGN (Storing Facts):
 *
 *   CREATE TABLE affiliate_sales (
 *     sale_id           SERIAL PRIMARY KEY,
 *     affiliate_code    VARCHAR(50) NOT NULL REFERENCES users(code),
 *     customer_email    VARCHAR(255) NOT NULL,
 *     purchase_date     TIMESTAMP NOT NULL,
 *     purchase_amount   DECIMAL(10,2) NOT NULL,
 *     commission_share  DECIMAL(5,2) NOT NULL,     -- e.g., 0.15
 *     commission_earned DECIMAL(10,2) NOT NULL,    -- computed at time of sale
 *     status            VARCHAR(20) NOT NULL DEFAULT 'PENDING'
 *   );
 *
 *   -- Har purchase ke liye: ek row insert karo
 *   INSERT INTO affiliate_sales
 *   (affiliate_code, customer_email, purchase_date,
 *    purchase_amount, commission_share, commission_earned, status)
 *   VALUES ('RAJ123', 'customer@example.com', NOW(),
 *           100.00, 0.15, 15.00, 'PAID');
 *
 *   -- Total earnings: facts se COMPUTE karo
 *   SELECT SUM(commission_earned) AS total_earnings
 *   FROM affiliate_sales
 *   WHERE affiliate_code = 'RAJ123'
 *     AND status = 'PAID';
 *
 * WHY THIS IS CORRECT:
 *   → Har row = ONE FACT (ek sale)
 *   → Earnings COMPUTED, not stored as pre-aggregated column
 *   → Audit possible: har sale traceable
 *   → Refund? → negative commission row add karo
 *   → Rule change? → commission_share at time of sale stored hai
 *   → Always verifiable: SUM(commission_earned) = truth
 */


/**
 * 17. WHAT IS A "TRUTH TABLE"? — Single Source of Truth
 * ───────────────────────────────────────────────────────
 * Hinglish:
 *   Truth Table = wo table jo ek entity ki SINGLE SOURCE OF TRUTH hai.
 *   Ek jagah, ek baar stored. Sab kuch wahi se reference karo.
 *
 * Examples:
 *   students(id, email, name, created_at)
 *     → Ek student = ek row. Student identity yahan define hoti hai.
 *     → Name change? Ek row update karo.
 *
 *   courses(id, code, title, price, instructor_id)
 *     → Ek course = ek row. Course ki definition yahan hai.
 *     → Title change? Ek row update karo.
 *
 *   payments(id, enrollment_id, amount, status, paid_at)
 *     → Ek payment transaction = ek row. Immutable once recorded.
 *
 * Properties of a Truth Table:
 *   ✓ Har row = ONE real-world fact
 *   ✓ Value change? EXACTLY ek jagah update karo
 *   ✓ Koi derived ya aggregated values stored nahi
 *   ✓ Sab attributes us entity ki direct properties hain
 *
 * SIGNAL OF A WRONG MODEL:
 *   Agar same value multiple rows mein update karni pade → model GALAT hai.
 *   → Student name 10 rows mein → WRONG
 *   → Course title 50,000 rows mein → WRONG
 *   → Commission rate multiple jagah → WRONG
 */


/**
 * 18. THE CONVENIENCE TABLE TRAP — Derive from Truth, Don't Replace It
 * ──────────────────────────────────────────────────────────────────────
 * Hinglish:
 *   Convenience tables reads fast karne ke liye banate hain.
 *   Ye GALAT nahi hain — lekin ye SIRF truth tables se derived hone chahiye.
 *   Kabhi bhi khud source of truth nahi banni chahiye.
 *
 * Examples of Convenience Tables:
 *   → Dashboard summary (pre-aggregated metrics)
 *   → Reporting tables (pre-joined data)
 *   → Materialized views
 *   → Cache tables
 *
 * CORRECT PATTERN:
 *
 *   ┌──────────────┐  ┌──────────────┐  ┌──────────────┐
 *   │  students    │  │  courses     │  │ enrollments  │
 *   │  (source)    │  │  (source)    │  │  (source)    │
 *   └──────┬───────┘  └──────┬───────┘  └──────┬───────┘
 *          └─────────────────┼─────────────────┘
 *                            ↓
 *          ┌─────────────────────────────────────────┐
 *          │   CONVENIENCE TABLE                     │
 *          │   (derived, refreshed periodically)     │
 *          │   dashboard_summary, reporting_snapshot │
 *          └─────────────────────────────────────────┘
 *
 * Rules for Convenience Tables:
 *   → ALWAYS derived from truth tables
 *   → Can be RECALCULATED from truth tables anytime
 *   → NEVER the only copy of data
 *   → Corrupt ho gayi? Truth se rebuild karo
 *   → Exist karte hain PERFORMANCE ke liye, DATA INTEGRITY ke liye nahi
 *
 * WHEN THEY BECOME DANGEROUS:
 *   → Jab wo khud source of truth ban jaayein
 *   → Jab unhe directly write karo bina truth update kiye
 *   → Jab tum unhe truth se recalculate na kar sako
 *
 * THE GOLDEN RULE:
 *   Truth tables define reality.
 *   Convenience tables derive FROM reality.
 *   Never let convenience OVERRIDE truth.
 */


In [ ]:
/**
 * Part-4: SCHEMA DESIGN — From Flat Tables to Production-Ready Models
 */

/**
 * 19. SCHEMA DESIGN — Tables, Constraints, and Referential Integrity
 * ───────────────────────────────────────────────────────────────────
 * Hinglish:
 *   Ab flat table ki jagah proper normalized schema banate hain.
 *   Har entity ki apni table, surrogate keys, constraints, bridge tables.
 *
 * 19.1 STUDENTS TABLE — Surrogate Key vs Natural Key
 * ────────────────────────────────────────────────────
 *
 *   CREATE TABLE students (
 *     student_id INT AUTO_INCREMENT PRIMARY KEY,   ← surrogate key
 *     email      VARCHAR(255) NOT NULL,
 *     full_name  VARCHAR(100) NOT NULL,
 *     created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
 *     CONSTRAINT uq_students_email UNIQUE (email)
 *   );
 *
 * WHY NOT USE EMAIL AS PRIMARY KEY?
 *   Email UNIQUE hai — haan. Lekin EMAIL STABLE NAHI HAI.
 *
 *   Agar email PK hoti:
 *     → Enrollments, payments, audit logs — sab mein FK hogi
 *     → Student email change kare → SABHIJAGAH update karo!
 *     → Update anomalies, lock contention, cascade nightmares
 *
 *   Surrogate key (student_id):
 *     ✓ STABLE — kabhi nahi badlega
 *     ✓ COMPACT — INT vs VARCHAR(255)
 *     ✓ Indexes ke liye efficient
 *     ✓ Foreign keys ke liye safe
 *
 *   Email UNIQUE hai — lekin ye ATTRIBUTE hai, IDENTITY nahi.
 *
 * 19.2 STUDENT PROFILES TABLE — Sparse Data Separation
 * ────────────────────────────────────────────────────
 *
 *   CREATE TABLE student_profiles (
 *     student_id      INT PRIMARY KEY,
 *     country_code    CHAR(2) NULL,
 *     graduation_year INT NULL,
 *     github_url      VARCHAR(255) NULL,
 *     FOREIGN KEY (student_id)
 *       REFERENCES students(student_id)
 *       ON DELETE CASCADE         ← profile ka koi matlab nahi student ke bina
 *       ON UPDATE RESTRICT
 *   );
 *
 * WHY SEPARATE TABLE?
 *   Har student GitHub, graduation year nahi provide karta.
 *   Agar students table mein rakho:
 *     → Zyada rows → NULL values → memory waste
 *     → Cache memory bhi waste
 *   PRODUCTION PRINCIPLE: Core identity tables LEAN rakho.
 *   Optional/sparse data → alag table.
 *
 * WHY ON DELETE CASCADE HERE?
 *   Profile ka koi meaning nahi student ke bina.
 *   Ye OWNERSHIP relationship hai → CASCADE use karo.
 *
 * 19.3 COURSES TABLE — Constraints and Referential Integrity
 * ──────────────────────────────────────────────────────────
 *
 *   CREATE TABLE courses (
 *     course_id     INT AUTO_INCREMENT PRIMARY KEY,
 *     course_code   VARCHAR(30) NOT NULL,
 *     title         VARCHAR(200) NOT NULL,
 *     instructor_id INT NOT NULL,
 *     base_price    DECIMAL(10,2) NOT NULL,
 *     currency      CHAR(3) NOT NULL,
 *     is_active     TINYINT NOT NULL DEFAULT 1,
 *
 *     CONSTRAINT uq_courses_code       UNIQUE (course_code),
 *     CONSTRAINT ck_courses_currency   CHECK (currency IN ('INR','USD')),
 *     CONSTRAINT ck_courses_price      CHECK (base_price >= 0),
 *
 *     CONSTRAINT fk_courses_instructor
 *       FOREIGN KEY (instructor_id) REFERENCES instructors(instructor_id)
 *       ON DELETE RESTRICT             ← instructor delete mat karo agar course hai
 *       ON UPDATE RESTRICT
 *   );
 *
 * WHY RESTRICT ON DELETE?
 *   Agar instructor delete ho aur course us par point kare:
 *     → Frontend joins fail
 *     → Reports break
 *     → Analytics corrupt
 *   RESTRICT force karta hai ki deletion intentional aur safe ho.
 *
 * PRODUCTION RULE:
 *   Application validation pe kabhi trust mat karo.
 *   Correctness DATABASE LAYER pe enforce karo.
 *
 * 19.4 ENROLLMENTS TABLE — Bridge Table for N:M
 * ──────────────────────────────────────────────
 *
 *   CREATE TABLE enrollments (
 *     enrollment_id INT AUTO_INCREMENT PRIMARY KEY,
 *     student_id    INT NOT NULL,
 *     course_id     INT NOT NULL,
 *     enrolled_at   DATETIME NOT NULL,
 *     status        VARCHAR(20) NOT NULL DEFAULT 'ACTIVE',
 *
 *     CONSTRAINT ck_enroll_status
 *       CHECK (status IN ('ACTIVE','COMPLETED','DROPPED')),
 *
 *     CONSTRAINT fk_enroll_student
 *       FOREIGN KEY (student_id) REFERENCES students(student_id)
 *       ON DELETE RESTRICT ON UPDATE RESTRICT,
 *
 *     CONSTRAINT fk_enroll_course
 *       FOREIGN KEY (course_id) REFERENCES courses(course_id)
 *       ON DELETE RESTRICT ON UPDATE RESTRICT,
 *
 *     -- Duplicate enrollment prevent karo
 *     CONSTRAINT uq_enroll_unique UNIQUE (student_id, course_id)
 *   );
 *
 * WHY UNIQUE(student_id, course_id)?
 *   Bina is constraint ke:
 *     → Student same course mein 3 baar enroll kar sakta hai
 *     → Data corruption + business rule violation
 *   CONSTRAINTS SILENT CORRUPTION PREVENT KARTE HAIN.
 *
 * 19.5 PAYMENTS TABLE — Financial Integrity
 * ──────────────────────────────────────────
 *
 *   CREATE TABLE payments (
 *     payment_pk INT AUTO_INCREMENT PRIMARY KEY,    ← internal ID
 *     payment_id VARCHAR(40) NOT NULL,              ← external gateway ID
 *     student_id INT NOT NULL,
 *     course_id  INT NOT NULL,
 *     amount     DECIMAL(10,2) NOT NULL,
 *     currency   CHAR(3) NOT NULL,
 *     status     VARCHAR(20) NOT NULL,
 *     created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
 *     paid_at    DATETIME NULL,
 *
 *     CONSTRAINT uq_payments_payment_id UNIQUE (payment_id),
 *     CONSTRAINT ck_pay_currency CHECK (currency IN ('INR','USD')),
 *     CONSTRAINT ck_pay_amount   CHECK (amount >= 0),
 *     CONSTRAINT ck_pay_status   CHECK (status IN ('PAID','PENDING','REFUNDED')),
 *
 *     CONSTRAINT fk_pay_student
 *       FOREIGN KEY (student_id) REFERENCES students(student_id)
 *       ON DELETE RESTRICT ON UPDATE RESTRICT,
 *
 *     CONSTRAINT fk_pay_course
 *       FOREIGN KEY (course_id) REFERENCES courses(course_id)
 *       ON DELETE RESTRICT ON UPDATE RESTRICT
 *   );
 *
 * WHY TWO IDs?
 *   payment_pk → apna internal database identifier
 *   payment_id → Razorpay/Stripe ka external reference
 *   External systems apna format change kar sakte hain →
 *   Apna stable identifier always rakho.
 *
 * WHY RESTRICT FOR PAYMENTS?
 *   Payments = FINANCIAL RECORDS.
 *   Student ya course delete nahi ho sakta agar payment history hai.
 *   Financial integrity > convenience.
 *   RESTRICT ensures auditability.
 */

In [ ]:
/**
 * Part-5: KEYS — Identity, Relationships, and Integrity
 */

/**
 * 20. KEYS — PK, FK, Composite, Surrogate, Natural
 * ──────────────────────────────────────────────────
 * Hinglish:
 *   Keys database ka backbone hain — bina keys ke relationships nahi, joins nahi,
 *   aur data integrity nahi.
 *
 * 20.1 PRIMARY KEY — Har Row ka Unique Identifier
 * ────────────────────────────────────────────────
 *   Rules:
 *     → UNIQUE hona zaroori (duplicate values nahi)
 *     → NOT NULL hona zaroori (har row mein value ho)
 *     → Ek table mein sirf EK primary key
 *
 *   CREATE TABLE students (
 *     student_id INT AUTO_INCREMENT PRIMARY KEY,   ← PK
 *     email      VARCHAR(255) NOT NULL,
 *     full_name  VARCHAR(100) NOT NULL
 *   );
 *
 *   WHY IT MATTERS:
 *     → Uniquely identifies every row
 *     → Reliable UPDATE aur DELETE possible
 *     → Foundation for foreign key relationships
 *     → Automatically ek index create karta hai
 *
 * 20.2 CANDIDATE KEY — Sab Columns Jo PK Ho Sakte Hain
 * ──────────────────────────────────────────────────────
 *   Candidate key = koi bhi column (ya combination) jo uniquely row identify kare.
 *   Ek choose hoti hai PRIMARY KEY → baaki ALTERNATE KEYS ban jaati hain.
 *
 *   students table mein:
 *   ┌────────────────┬───────────────────────────────────┐
 *   │ Column         │ Candidate key ban sakta hai?      │
 *   ├────────────────┼───────────────────────────────────┤
 *   │ student_id     │ YES (unique, never changes)       │
 *   │ email          │ YES (unique in business domain)   │
 *   │ full_name      │ NO (duplicates possible)          │
 *   │ phone          │ YES (if unique)                   │
 *   └────────────────┴───────────────────────────────────┘
 *
 *   PRIMARY KEY  = student_id (chosen)
 *   ALTERNATE KEY = email (UNIQUE constraint se enforce)
 *
 * 20.3 UNIQUE CONSTRAINT — Business Rule, Not Identity
 * ─────────────────────────────────────────────────────
 *   Hinglish:
 *     UNIQUE = "ye column mein duplicate values nahi honge."
 *     PRIMARY KEY se alag hai — ye identity nahi deta, business rule enforce karta hai.
 *
 *   ALTER TABLE students ADD CONSTRAINT uq_email UNIQUE (email);
 *
 *   Difference:
 *     PRIMARY KEY = identity (row ko kaise refer karein)
 *     UNIQUE = business rule (duplicate emails nahi)
 *
 * 20.4 FOREIGN KEY — Valid Relationships Enforce Karo
 * ────────────────────────────────────────────────────
 *   Hinglish:
 *     FK ensure karta hai ki ek table mein stored reference valid hai.
 *     Bina FK ke: orphan records possible (payment without student).
 *     FK ke saath: DB khud validate karta hai.
 *
 *   CREATE TABLE payments (
 *     payment_id INT AUTO_INCREMENT PRIMARY KEY,
 *     student_id INT NOT NULL,
 *     amount     DECIMAL(10,2) NOT NULL,
 *     FOREIGN KEY (student_id) REFERENCES students(student_id)
 *   );
 *
 *   ┌──────────────────────────────────┬────────────────────────────────┐
 *   │ Action                           │ Result                         │
 *   ├──────────────────────────────────┼────────────────────────────────┤
 *   │ Insert payment, student_id = 1   │ OK (student exists)            │
 *   │ Insert payment, student_id = 99  │ ERROR! Student doesn't exist   │
 *   │ Delete student with payments     │ RESTRICT or CASCADE (decided)  │
 *   └──────────────────────────────────┴────────────────────────────────┘
 *
 * 20.5 COMPOSITE KEY — Multiple Columns as Primary Key
 * ──────────────────────────────────────────────────────
 *   Hinglish:
 *     Kabhi kabhi ek column uniqueness guarantee nahi karta.
 *     Composite key = multiple columns combine karke ek unique key banao.
 *
 *   CREATE TABLE enrollments (
 *     student_id  INT NOT NULL,
 *     course_id   INT NOT NULL,
 *     enrolled_at DATETIME NOT NULL,
 *     PRIMARY KEY (student_id, course_id)   ← composite key
 *   );
 *
 *   → student_id 1, course_id 101 → OK
 *   → student_id 1, course_id 102 → OK (different course)
 *   → student_id 1, course_id 101 → REJECTED! (same combo)
 *
 *   WHEN TO USE: Junction/bridge tables for N:M relationships.
 *
 * 20.6 SURROGATE vs NATURAL KEY
 * ────────────────────────────────
 *   Hinglish:
 *     Natural key = business mein meaning rakhti hai (email, phone, PAN)
 *     Surrogate key = DB ke liye generate ki gayi ID, koi business meaning nahi
 *
 *   ┌─────────────────┬────────────────────┬────────────────────────────────┐
 *   │                 │ NATURAL KEY        │ SURROGATE KEY                  │
 *   ├─────────────────┼────────────────────┼────────────────────────────────┤
 *   │ Example         │ email, phone, PAN  │ student_id (auto-increment)    │
 *   │ Pros            │ Business meaning   │ Stable (never changes)         │
 *   │                 │ No extra column    │ Small (INT vs VARCHAR)         │
 *   │ Cons            │ Can CHANGE         │ No business meaning            │
 *   │                 │ Larger (VARCHAR)   │ Extra column                   │
 *   └─────────────────┴────────────────────┴────────────────────────────────┘
 *
 * WHY INDUSTRY PREFERS SURROGATE KEYS:
 *
 *   REASON 1: STABILITY
 *     student_id = 1 → KABHI NAHI BADLEGA
 *     Email change karo → sirf students table mein ek update
 *     Foreign keys unaffected!
 *
 *   REASON 2: PERFORMANCE
 *     INT/BIGINT = 4-8 bytes
 *     VARCHAR(255) = max 255 bytes
 *     → Faster indexing, efficient joins, more B+ tree entries per page
 *
 *   REASON 3: ISOLATION
 *     Business rules change hoti hain — email format, phone format.
 *     Surrogate keys immune hain.
 *
 * IMPORTANT CLARIFICATION:
 *   Surrogate key akele duplicate prevent nahi karta.
 *   Complete pattern chahiye:
 *     → Surrogate key for IDENTITY (PRIMARY KEY)
 *     → UNIQUE constraint for BUSINESS RULES
 *
 *   CREATE TABLE students (
 *     student_id INT AUTO_INCREMENT PRIMARY KEY,  ← identity
 *     email      VARCHAR(255) NOT NULL UNIQUE,    ← business rule
 *     full_name  VARCHAR(100) NOT NULL
 *   );
 *   Dono zaroori hain. Dono alag kaam karte hain.
 */

In [ ]:
/**
 * Part-6: NORMALIZATION DEEP DIVE — 1NF, 2NF, 3NF
 */

/**
 * 21. NORMALIZATION — Store Each Fact Once
 * ─────────────────────────────────────────
 * Hinglish:
 *   Normalization ek process hai jisme DB tables is tarah design karte hain
 *   ki har real-world fact EXACTLY ONCE stored ho, baaki sab keys se reference karo.
 *   Ye ek MINDSET hai — rules yaad karne ki nahi, TRUTH PROTECT karne ki baat hai.
 *
 * When you follow normalization:
 *   → Schema STABLE rehta hai
 *   → Data CONSISTENT rehta hai
 *   → Anomalies PREVENTED hote hain
 *
 * When you ignore it:
 *   → Update anomaly (kuch rows update, baaki miss)
 *   → Insert anomaly (redundant data chahiye)
 *   → Delete anomaly (galat data chala jaata hai)
 */


/**
 * 22. DATA ANOMALIES — Update, Insert, Delete
 * ─────────────────────────────────────────────
 * Hinglish:
 *   Denormalized tables mein teen tarah ki anomalies hoti hain.
 *   Samajhna zaroori hai kyunki yahi normalization ka REASON hai.
 *
 * 22.1 UPDATE ANOMALY — Duplicate Data = Inconsistent Updates
 * ─────────────────────────────────────────────────────────────
 *
 *   BAD DESIGN (violates normalization):
 *
 *   enrollments table:
 *   ┌────────────┬───────────────┬───────────┬─────────────────┐
 *   │ student_id │ student_name  │ course_id │ enrollment_date │
 *   ├────────────┼───────────────┼───────────┼─────────────────┤
 *   │ 1          │ Asha Sharma   │ 101       │ 2026-01-01      │
 *   │ 1          │ Asha S.       │ 102       │ 2026-01-05      │   ← naam alag!
 *   │ 2          │ Ravi Kumar    │ 101       │ 2026-01-02      │
 *   └────────────┴───────────────┴───────────┴─────────────────┘
 *
 *   Asha apna naam update karti hai:
 *     UPDATE enrollments SET student_name = 'Asha Sharma'
 *     WHERE student_id = 1;
 *   Koi ek row miss ho gayi → inconsistency → reports galat → analytics break
 *
 *   NORMALIZED SOLUTION:
 *     students table: student_name ek row mein
 *     enrollments table: sirf student_id store karo
 *     Name update? → ek row update, sab enrollments automatically correct
 *
 * 22.2 INSERT ANOMALY — Duplicate Data Makes Inserts Complex
 * ───────────────────────────────────────────────────────────
 *
 *   BAD DESIGN: enrollments mein student_name, course_name, instructor_name sab hai.
 *   New enrollment insert karo:
 *
 *   INSERT INTO enrollments VALUES (
 *     3, 'Neha Singh', 101, 'SQL Basics', 'Raj Kumar', ...
 *   );
 *
 *   'Neha Singh' vs 'Neha S.' → inconsistency
 *   'SQL Basics' vs 'SQL Fundamentals' → confusion
 *   Naya column add hua → sab insert statements change karo
 *
 *   NORMALIZED SOLUTION:
 *     INSERT INTO students (student_id, name) VALUES (3, 'Neha');
 *     INSERT INTO enrollments (student_id, course_id, enrolled_at)
 *     VALUES (3, 101, NOW());
 *     No duplication. Simple. Error-free.
 *
 * 22.3 DELETE ANOMALY — Unclear Relationships = Data Loss
 * ─────────────────────────────────────────────────────────
 *
 *   BAD DESIGN: enrollments mein sab kuch — student info, course info, payment.
 *
 *   Student ek course se withdraw karta hai:
 *     DELETE FROM enrollments WHERE student_id = 1 AND course_id = 101;
 *
 *   Agar ye student ki ONLY enrollment thi:
 *     → Student record completely gone!
 *     → All history lost
 *     → Financial audit impossible
 *
 *   NORMALIZED SOLUTION:
 *     DELETE FROM enrollments WHERE student_id = 1 AND course_id = 101;
 *     → Sirf enrollment delete hua
 *     → Student still exists in students table
 *     → Course still exists in courses table
 *     → Payments still in payments table
 *     → FK with ON DELETE RESTRICT → accidental loss prevent hoti hai
 */


/**
 * 23. NORMAL FORMS — 1NF, 2NF, 3NF
 * ───────────────────────────────────
 * Hinglish:
 *   Normal forms ek step-by-step framework hain data organize karne ka.
 *   Sab se pehle 1NF satisfy karo, phir 2NF, phir 3NF.
 *
 * 23.1 FIRST NORMAL FORM (1NF) — Atomic Values, No Repeating Groups
 * ───────────────────────────────────────────────────────────────────
 *   Rules:
 *     1. Har column mein SINGLE, ATOMIC value ho
 *        → Lists nahi, comma-separated values nahi
 *     2. Repeating groups nahi (course1, course2, course3 columns)
 *     3. Har row ek entity ka ek record represent kare
 *
 *   BAD (violates 1NF):
 *   ┌────────────┬─────────────────┐
 *   │ student_id │ course_ids      │
 *   ├────────────┼─────────────────┤
 *   │ 1          │ 2,4,6           │  ← list! parse karna padega
 *   └────────────┴─────────────────┘
 *   → Indexing useless, filtering slow, joins break
 *
 *   CORRECT (1NF compliant):
 *   ┌────────────┬───────────┐
 *   │ student_id │ course_id │
 *   ├────────────┼───────────┤
 *   │ 1          │ 2         │
 *   │ 1          │ 4         │
 *   │ 1          │ 6         │
 *   └────────────┴───────────┘
 *   → Indexing works, instant search, clean joins
 *
 *   REPEATING GROUPS BAD DESIGN:
 *   ┌────────────┬─────────────────┬─────────────────┬─────────────────┐
 *   │ student_id │ course1_notes   │ course2_notes   │ course3_notes   │
 *   ├────────────┼─────────────────┼─────────────────┼─────────────────┤
 *   │ 1          │ 'Learned JOINs' │ 'Learned INDEX' │ NULL            │
 *   └────────────┴─────────────────┴─────────────────┴─────────────────┘
 *   → Max 3 courses assume karna, 4th course aaye toh schema change karo!
 *
 *   CORRECT:
 *   ┌────────────┬───────────┬─────────────────────┐
 *   │ note_id    │ student_id│ note_text           │
 *   ├────────────┼───────────┼─────────────────────┤
 *   │ 1          │ 1         │ 'Learned JOINs'     │
 *   │ 2          │ 1         │ 'Learned INDEX'     │
 *   └────────────┴───────────┴─────────────────────┘
 *   → Unlimited notes, stable schema, clean queries
 *
 *   1NF IS THE BARE MINIMUM.
 *   Without it: indexes fail, searches slow, joins unreliable.
 *
 * 23.2 SECOND NORMAL FORM (2NF) — No Partial Dependencies
 * ─────────────────────────────────────────────────────────
 *   Applies ONLY to tables with COMPOSITE PRIMARY KEYS.
 *
 *   Rule:
 *     → Already in 1NF
 *     → Har non-key column PURI composite key pe depend kare, partial nahi
 *
 *   WHAT IS PARTIAL DEPENDENCY?
 *
 *   BAD DESIGN (violates 2NF):
 *   PRIMARY KEY = (student_id, course_id)
 *
 *   ┌────────────┬───────────┬─────────────────┬─────────────────┐
 *   │ student_id │ course_id │ student_name    │ enrollment_date │
 *   ├────────────┼───────────┼─────────────────┼─────────────────┤
 *   │ 1          │ 101       │ Asha Sharma     │ 2026-01-01      │
 *   │ 1          │ 102       │ Asha Sharma     │ 2026-01-05      │   ← naam repeat
 *   │ 2          │ 101       │ Ravi Kumar      │ 2026-01-02      │
 *   └────────────┴───────────┴─────────────────┴─────────────────┘
 *
 *   → student_name depends ONLY on student_id (partial!)
 *   → enrollment_date depends on FULL key (student + course) ← sahi hai
 *   → Ye PARTIAL DEPENDENCY hai → 2NF violate hoti hai
 *
 *   CORRECT (2NF compliant):
 *
 *   students table:
 *   ┌────────────┬───────────────┐
 *   │ student_id │ student_name  │
 *   ├────────────┼───────────────┤
 *   │ 1          │ Asha Sharma   │   ← ek jagah
 *   │ 2          │ Ravi Kumar    │
 *   └────────────┴───────────────┘
 *
 *   enrollments table:
 *   ┌────────────┬───────────┬─────────────────┐
 *   │ student_id │ course_id │ enrollment_date │
 *   ├────────────┼───────────┼─────────────────┤
 *   │ 1          │ 101       │ 2026-01-01      │
 *   │ 1          │ 102       │ 2026-01-05      │
 *   │ 2          │ 101       │ 2026-01-02      │
 *   └────────────┴───────────┴─────────────────┘
 *
 *   RULE: Har non-key column WHOLE key pe depend kare, kisi part pe nahi.
 *   NOTE: SINGLE column PK wali table automatically 2NF satisfy karti hai.
 *
 * 23.3 THIRD NORMAL FORM (3NF) — No Transitive Dependencies
 * ───────────────────────────────────────────────────────────
 *   Rule:
 *     → Already in 2NF
 *     → Non-key columns kisi DOOSRE non-key column pe depend nahi karein
 *
 *   WHAT IS TRANSITIVE DEPENDENCY?
 *
 *   BAD DESIGN (violates 3NF):
 *   PRIMARY KEY = (student_id, course_id)
 *
 *   ┌────────────┬───────────┬────────────┬─────────────────┐
 *   │ student_id │ course_id │ payment_id │ payment_amount  │
 *   ├────────────┼───────────┼────────────┼─────────────────┤
 *   │ 1          │ 101       │ pay_1001   │ 499.00          │
 *   │ 1          │ 102       │ pay_1002   │ 799.00          │
 *   └────────────┴───────────┴────────────┴─────────────────┘
 *
 *   DEPENDENCY CHAIN:
 *     (student_id, course_id) → payment_id → payment_amount
 *
 *   payment_amount directly PK pe depend nahi karta — payment_id pe karta hai.
 *   Ye TRANSITIVE DEPENDENCY hai → 3NF violate hoti hai.
 *
 *   CORRECT (3NF compliant):
 *
 *   payments table:
 *   ┌────────────┬─────────────────┬────────────┐
 *   │ payment_id │ enrollment_id   │ amount     │
 *   ├────────────┼─────────────────┼────────────┤
 *   │ pay_1001   │ 1               │ 499.00     │
 *   │ pay_1002   │ 2               │ 799.00     │
 *   └────────────┴─────────────────┴────────────┘
 *
 *   → amount sirf apne PK (payment_id) pe depend karta hai
 *   → Koi transitive dependency nahi
 *
 *   RULE: Non-key columns depend ONLY on the key, nothing else.
 *   3NF = standard for MOST PRODUCTION DATABASES.
 */

